# MindSight — Complete Fusion Pipeline (Final)
**All 4 best models integrated and tested with real inputs**

| Modality | Model | Dataset | Accuracy |
|---|---|---|---|
| Facial | MobileNetV2 + Dense Head | Balanced FER (45,304 images) | 84.60% |
| Voice | YAMNet + Dense Network | TESS (2,800 audio files) | 96.96% |
| Text | BERT fine-tuned | dair-ai/emotion + GoEmotions | See v3 |
| Behavioral | Random Forest | ScreenTime + Digital + Mobile | 91.25% |

### Instructions
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order
3. In the last cell — enter YOUR real inputs and see the output

In [1]:
# ============================================================
# CELL 1 — Mount Drive & Install Libraries
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers torch librosa tensorflow tensorflow_hub scikit-learn groq -q
print('✅ All libraries installed!')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 4.8 MB/s eta 0:00:00
✅ All libraries installed!


In [2]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import os, json, pickle, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import tensorflow as tf
from tensorflow import keras
import tensorflow_hub as hub
import librosa
from transformers import BertTokenizer, BertForSequenceClassification
warnings.filterwarnings('ignore')

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = '/content/drive/MyDrive/MindSight_Models'

# Shared emotion labels — all models output these
EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

# Fusion weights from MindLift paper
WEIGHTS = {'facial': 0.40, 'voice': 0.30, 'text': 0.20, 'behavioral': 0.10}

print(f'✅ Device: {DEVICE}')
print(f'✅ Emotions: {EMOTIONS}')
print(f'✅ Weights: {WEIGHTS}')

✅ Device: cpu
✅ Emotions: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
✅ Weights: {'facial': 0.4, 'voice': 0.3, 'text': 0.2, 'behavioral': 0.1}


In [3]:
# ============================================================
# CELL 3 — Load Facial Model (MobileNetV2, 84.60%)
# ============================================================
print('Loading facial model...')
facial_model = keras.models.load_model(f'{SAVE_DIR}/facial_model.keras')

with open(f'{SAVE_DIR}/facial_labels.json') as f:
    facial_labels = json.load(f)

print(f'✅ Facial model loaded!')
print(f'   Labels: {facial_labels}')

# Quick test
dummy_face = np.random.rand(1, 96, 96, 3).astype('float32')
test_out   = facial_model.predict(dummy_face, verbose=0)[0]
print(f'   Output shape: {test_out.shape} | Sum: {test_out.sum():.4f} ✅')

Loading facial model...
✅ Facial model loaded!
   Labels: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
   Output shape: (7,) | Sum: 1.0000 ✅


In [4]:
# ============================================================
# CELL 4 — Load Voice Model (YAMNet + Dense, 96.96%)
# ============================================================
print('Loading YAMNet from TensorFlow Hub...')
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')
print('✅ YAMNet loaded!')

print('Loading voice model...')
voice_model = keras.models.load_model(f'{SAVE_DIR}/voice_model.keras')

with open(f'{SAVE_DIR}/voice_labels.json') as f:
    voice_labels = [str(l) for l in json.load(f)]

print(f'✅ Voice model loaded!')
print(f'   Labels: {voice_labels}')

# Quick test
dummy_audio = np.random.rand(1, 1024).astype('float32')
test_out    = voice_model.predict(dummy_audio, verbose=0)[0]
print(f'   Output shape: {test_out.shape} | Sum: {test_out.sum():.4f} ✅')

Loading YAMNet from TensorFlow Hub...
✅ YAMNet loaded!
Loading voice model...
✅ Voice model loaded!
   Labels: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
   Output shape: (7,) | Sum: 1.0000 ✅


In [5]:
# ============================================================
# CELL 5 — Load BERT Text Model (v3)
# ============================================================
BERT_DIR = f'{SAVE_DIR}/bert_text_model_v3'

print('Loading BERT tokenizer and model...')
bert_tokenizer = BertTokenizer.from_pretrained(BERT_DIR)
bert_model     = BertForSequenceClassification.from_pretrained(BERT_DIR)
bert_model     = bert_model.to(DEVICE)
bert_model.eval()

with open(f'{BERT_DIR}/text_labels.json') as f:
    text_labels = json.load(f)

print(f'✅ BERT model loaded on {DEVICE}!')
print(f'   Labels: {text_labels}')

# Quick test
def predict_bert(text):
    enc = bert_tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=128, padding='max_length'
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = bert_model(**enc).logits
    return F.softmax(logits, dim=1).squeeze().cpu().numpy()

test_probs = predict_bert('I feel happy today')
print(f'   Test prediction: {text_labels[np.argmax(test_probs)]} ✅')

Loading BERT tokenizer and model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ BERT model loaded on cpu!
   Labels: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
   Test prediction: happy ✅


In [6]:
# ============================================================
# CELL 6 — Load Behavioral Model (Random Forest, 91.25%)
# ============================================================
print('Loading behavioral model...')
with open(f'{SAVE_DIR}/behavioral_model.pkl', 'rb') as f:
    beh_model = pickle.load(f)

with open(f'{SAVE_DIR}/behavioral_labels.json') as f:
    beh_labels = json.load(f)

with open(f'{SAVE_DIR}/behavioral_columns.json') as f:
    beh_columns = json.load(f)

print(f'✅ Behavioral model loaded!')
print(f'   Stress labels: {beh_labels}')
print(f'   Input columns: {beh_columns}')

Loading behavioral model...
✅ Behavioral model loaded!
   Stress labels: ['high', 'low', 'medium']
   Input columns: ['age', 'gender', 'occupation', 'work_mode', 'screen_time_hours', 'work_screen_hours', 'leisure_screen_hours', 'sleep_hours', 'sleep_quality_1_5', 'productivity_0_100', 'exercise_minutes_per_week', 'social_hours_per_week']


In [8]:
# ============================================================
# CELL 7 — All 4 Models Loaded!
# ============================================================
print('=' * 55)
print('  ALL 4 MODELS LOADED SUCCESSFULLY')
print('=' * 55)
print(f'  Facial     : MobileNetV2 + Dense  | 84.60%')
print(f'  Voice      : YAMNet + Dense       | 96.96%')
print(f'  Text       : BERT fine-tuned      | 82.35%')
print(f'  Behavioral : Random Forest        | 91.25%')
print('=' * 55)

  ALL 4 MODELS LOADED SUCCESSFULLY
  Facial     : MobileNetV2 + Dense  | 84.60%
  Voice      : YAMNet + Dense       | 96.96%
  Text       : BERT fine-tuned      | 82.35%
  Behavioral : Random Forest        | 91.25%


In [9]:
# ============================================================
# CELL 8 — Helper Functions
# ============================================================

def align_to_emotions(probs, source_labels, target_labels):
    """Map any model's probability output to shared 7-emotion vector"""
    aligned = np.zeros(len(target_labels))
    for i, label in enumerate(source_labels):
        label = str(label).lower()
        if label in target_labels:
            j = target_labels.index(label)
            aligned[j] = probs[i]
    total = aligned.sum()
    if total > 0:
        aligned = aligned / total
    return aligned


def behavioral_to_emotion(stress_probs, beh_labels, target_labels):
    """Convert stress level (low/medium/high) → emotion probability vector"""
    label_list  = [str(l).lower() for l in beh_labels]
    high_prob   = stress_probs[label_list.index('high')]   if 'high'   in label_list else 0
    medium_prob = stress_probs[label_list.index('medium')] if 'medium' in label_list else 0
    low_prob    = stress_probs[label_list.index('low')]    if 'low'    in label_list else 0

    stress_map = np.zeros(len(target_labels))
    for i, emotion in enumerate(target_labels):
        if emotion == 'sad':
            stress_map[i] = high_prob * 0.4 + medium_prob * 0.3
        elif emotion == 'fear':
            stress_map[i] = high_prob * 0.3
        elif emotion == 'angry':
            stress_map[i] = high_prob * 0.3
        elif emotion == 'neutral':
            stress_map[i] = medium_prob * 0.5 + low_prob * 0.3
        elif emotion == 'happy':
            stress_map[i] = low_prob * 0.7
        else:
            stress_map[i] = 0.0

    total = stress_map.sum()
    if total > 0:
        stress_map = stress_map / total
    return stress_map


def get_facial_probs(image_array):
    """image_array: numpy array shape (96, 96, 3), values 0-1"""
    inp  = np.expand_dims(image_array, axis=0).astype('float32')
    probs = facial_model.predict(inp, verbose=0)[0]
    return align_to_emotions(probs, facial_labels, EMOTIONS)


def get_voice_probs(audio_path):
    """audio_path: path to .wav file"""
    audio, sr = librosa.load(audio_path, sr=16000)
    _, embeddings, _ = yamnet(audio)
    embedding = np.mean(embeddings.numpy(), axis=0).reshape(1, -1)
    probs     = voice_model.predict(embedding, verbose=0)[0]
    return align_to_emotions(probs, voice_labels, EMOTIONS)


def get_text_probs(text):
    """text: any string"""
    probs = predict_bert(text)
    return align_to_emotions(probs, text_labels, EMOTIONS)


def get_behavioral_probs(behavioral_dict):
    """behavioral_dict: dictionary of behavioral features"""
    df        = pd.DataFrame([behavioral_dict])
    df        = df.reindex(columns=beh_columns, fill_value=0)
    stress_probs = beh_model.predict_proba(df)[0]
    return behavioral_to_emotion(stress_probs, beh_labels, EMOTIONS)


print('✅ All helper functions ready!')

✅ All helper functions ready!


In [10]:
# ============================================================
# CELL 9 — Main Fusion Function
# ============================================================
def fuse(facial_probs, voice_probs, text_probs, behavioral_probs):
    """Weighted average fusion of all 4 modalities"""
    fused = (
        WEIGHTS['facial']     * facial_probs +
        WEIGHTS['voice']      * voice_probs +
        WEIGHTS['text']       * text_probs +
        WEIGHTS['behavioral'] * behavioral_probs
    )
    final_emotion = EMOTIONS[np.argmax(fused)]
    confidence    = round(float(np.max(fused)) * 100, 2)

    return {
        'final_emotion':    final_emotion,
        'confidence':       confidence,
        'fused_scores':     dict(zip(EMOTIONS, fused.tolist())),
        'modality_scores':  {
            'facial':     dict(zip(EMOTIONS, facial_probs.tolist())),
            'voice':      dict(zip(EMOTIONS, voice_probs.tolist())),
            'text':       dict(zip(EMOTIONS, text_probs.tolist())),
            'behavioral': dict(zip(EMOTIONS, behavioral_probs.tolist())),
        }
    }


def display_result(result, user_text=''):
    """Pretty print the fusion result"""
    emotion_emoji = {
        'angry': '😠', 'disgust': '🤢', 'fear': '😨',
        'happy': '😊', 'neutral': '😐', 'sad': '😢', 'surprise': '😲'
    }
    e = result['final_emotion']

    print('\n' + '=' * 55)
    print('  MINDSIGHT — EMOTION DETECTION RESULT')
    print('=' * 55)
    if user_text:
        print(f'  Text input : "{user_text[:50]}"')
    print(f'  {emotion_emoji.get(e,"")} Final emotion : {e.upper()}')
    print(f'  Confidence  : {result["confidence"]}%')
    print('─' * 55)

    print('  Fused emotion scores:')
    for emotion, score in sorted(result['fused_scores'].items(), key=lambda x: -x[1]):
        bar  = '█' * int(score * 30)
        emoj = emotion_emoji.get(emotion, '')
        print(f'    {emoj} {emotion:<10} {score:.4f}  {bar}')

    print('─' * 55)
    print('  Per-modality predictions:')
    for modality, scores in result['modality_scores'].items():
        top    = max(scores, key=scores.get)
        weight = WEIGHTS[modality]
        emoj   = emotion_emoji.get(top, '')
        print(f'    {modality:<12} → {emoj} {top:<10} (weight={weight})')
    print('=' * 55)


print('✅ Fusion function ready!')

✅ Fusion function ready!


In [14]:
# ============================================================
# CELL 10 — Groq Report Generator
# ============================================================
from groq import Groq

# ⚠️ PASTE YOUR GROQ API KEY HERE
GROQ_API_KEY = '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'
client       = Groq(api_key=GROQ_API_KEY)

def generate_report(result, user_text, behavioral_input):
    emotion    = result['final_emotion']
    confidence = result['confidence']
    scores     = result['fused_scores']
    top3       = sorted(scores.items(), key=lambda x: -x[1])[:3]
    top_str    = ', '.join([f"{e} ({round(s*100)}%)" for e, s in top3])

    prompt = f"""You are a compassionate mental health support assistant for students.

A student's multimodal AI analysis detected:
- Primary emotion: {emotion.upper()} (confidence: {confidence}%)
- Top emotions: {top_str}
- Student wrote: "{user_text}"
- Sleep hours: {behavioral_input.get('sleep_hours', 'unknown')}
- Screen time: {behavioral_input.get('screen_time_hours', 'unknown')} hours
- Exercise per week: {behavioral_input.get('exercise_minutes_per_week', 'unknown')} mins
- Productivity score: {behavioral_input.get('productivity_0_100', 'unknown')}/100

Write a warm, personalized mental health report with exactly these 4 sections:
1. WHAT WE DETECTED (2 sentences — explain what AI found in simple words)
2. WHAT THIS MEANS (2 sentences — what might be causing this emotionally)
3. 3 THINGS TO DO TONIGHT (3 specific actionable tips for a student)
4. ENCOURAGING NOTE (1 warm closing sentence)

Tone: warm, non-clinical, supportive. Max 150 words."""

    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=350,
        temperature=0.7
    )
    return response.choices[0].message.content


print('✅ Report generator ready!')
print('⚠️  Make sure to paste your Groq API key above before running Cell 13')

✅ Report generator ready!
⚠️  Make sure to paste your Groq API key above before running Cell 13


In [16]:
# ============================================================
# CELL 11 — Test with IMAGE URL (Real Facial Input)
# ============================================================
import requests
from PIL import Image
from io import BytesIO

def load_face_from_url(url):
    """Load face image from URL → numpy array (96,96,3)"""
    response = requests.get(url)
    img      = Image.open(BytesIO(response.content)).convert('RGB').resize((96, 96))
    return np.array(img) / 255.0

def load_face_from_upload():
    """Upload face image from your computer"""
    from google.colab import files
    uploaded = files.upload()
    fname    = list(uploaded.keys())[0]
    img      = Image.open(fname).convert('RGB').resize((96, 96))
    return np.array(img) / 255.0

# ── TEST with a real face image URL ──
# You can replace this URL with any face photo URL
# Or comment this out and use load_face_from_upload() instead
FACE_URL = 'https://img.freepik.com/premium-photo/angry-man-yelling-white-background_861973-35705.jpg?w=2000'

# Try loading — if URL fails, use dummy
try:
    face_array = load_face_from_url(FACE_URL)
    facial_probs = get_facial_probs(face_array)
    top_face = EMOTIONS[np.argmax(facial_probs)]
    print(f'✅ Facial prediction: {top_face.upper()} ({max(facial_probs)*100:.1f}%)')
    print(f'   All scores: {dict(zip(EMOTIONS, [round(p,3) for p in facial_probs]))}')
except Exception as e:
    print(f'⚠️  Could not load image URL: {e}')
    print('   Using dummy face input for now')
    facial_probs = align_to_emotions(
        facial_model.predict(np.random.rand(1,96,96,3).astype("float32"), verbose=0)[0],
        facial_labels, EMOTIONS
    )

print('\n💡 To use YOUR face photo:')
print('   face_array   = load_face_from_upload()')
print('   facial_probs = get_facial_probs(face_array)')

✅ Facial prediction: ANGRY (94.4%)
   All scores: {'angry': np.float64(0.944), 'disgust': np.float64(0.0), 'fear': np.float64(0.004), 'happy': np.float64(0.001), 'neutral': np.float64(0.0), 'sad': np.float64(0.051), 'surprise': np.float64(0.0)}

💡 To use YOUR face photo:
   face_array   = load_face_from_upload()
   facial_probs = get_facial_probs(face_array)


In [18]:
# ── Define the upload function first ──
from google.colab import files
import numpy as np

def load_audio_from_upload():
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    return fname

# ── OPTION A: Upload your own .wav file ──
audio_path  = load_audio_from_upload()
voice_probs = get_voice_probs(audio_path)

print(f'✅ Voice prediction: {EMOTIONS[np.argmax(voice_probs)].upper()}')

Saving audio.wav to audio.wav
✅ Voice prediction: SAD


In [20]:
# ============================================================
# CELL 13 — REAL WORLD TEST
# ✏️  EDIT THESE INPUTS WITH YOUR REAL DATA
# ============================================================

# ── 1. YOUR TEXT ─────────────────────────────────────────────
# Write how you're feeling right now
user_text = "I am so stressed about my exams, I'm very upset I cannot sleep and feel completely overwhelmed"

# ── 2. YOUR BEHAVIORAL DATA ──────────────────────────────────
# Fill in your actual values
behavioral_input = {
    'age':                       21,
    'gender':                    0,      # 0=Female, 1=Male
    'occupation':                2,      # 2=Student
    'work_mode':                 1,
    'screen_time_hours':         7.5,    # total daily screen time
    'work_screen_hours':         7.0,    # screen time for study/work
    'leisure_screen_hours':      5.5,    # screen time for entertainment
    'sleep_hours':               5.0,    # hours slept last night
    'sleep_quality_1_5':         2,      # 1=very poor, 5=excellent
    'productivity_0_100':        35,     # how productive you felt (0-100)
    'exercise_minutes_per_week': 20,     # exercise in mins per week
    'social_hours_per_week':     2.0,    # hours socializing per week
}

# ── RUN PIPELINE ─────────────────────────────────────────────
print('Running MindSight pipeline...\n')

# Get text probs (real)
text_probs         = get_text_probs(user_text)
# Get behavioral probs (real)
behavioral_probs   = get_behavioral_probs(behavioral_input)
# facial_probs and voice_probs already set above

# Fuse all 4
result = fuse(facial_probs, voice_probs, text_probs, behavioral_probs)

# Display results
display_result(result, user_text)

# Generate AI report
print('\nGenerating personalized mental health report...')
try:
    report = generate_report(result, user_text, behavioral_input)
    print('\n' + '=' * 55)
    print('  PERSONALIZED MENTAL HEALTH REPORT')
    print('=' * 55)
    print(report)
    print('=' * 55)
except Exception as e:
    print(f'⚠️  Groq report failed: {e}')
    print('   Make sure you pasted your Groq API key in Cell 10')

Running MindSight pipeline...


  MINDSIGHT — EMOTION DETECTION RESULT
  Text input : "I am so stressed about my exams, I'm very upset I "
  😠 Final emotion : ANGRY
  Confidence  : 40.9%
───────────────────────────────────────────────────────
  Fused emotion scores:
    😠 angry      0.4090  ████████████
    😢 sad        0.3634  ██████████
    😲 surprise   0.1333  ███
    😨 fear       0.0832  ██
    😐 neutral    0.0045  
    😊 happy      0.0039  
    🤢 disgust    0.0028  
───────────────────────────────────────────────────────
  Per-modality predictions:
    facial       → 😠 angry      (weight=0.4)
    voice        → 😢 sad        (weight=0.3)
    text         → 😲 surprise   (weight=0.2)
    behavioral   → 😢 sad        (weight=0.1)

Generating personalized mental health report...

  PERSONALIZED MENTAL HEALTH REPORT
## Step 1: Explain the AI findings in simple words
Our analysis detected that you're feeling angry and sad, with a significant amount of stress about your exams. This mix of 

In [21]:
# ============================================================
# CELL 14 — Test Multiple Scenarios
# See how MindSight responds to different student situations
# ============================================================

scenarios = [
    {
        'name': 'Stressed student before exams',
        'text': 'I cannot sleep, my exams are tomorrow and I feel like I will fail everything',
        'behavioral': {
            'sleep_hours': 4.0, 'screen_time_hours': 10.0,
            'productivity_0_100': 25, 'exercise_minutes_per_week': 0,
            'social_hours_per_week': 1.0, 'sleep_quality_1_5': 1,
            'age': 20, 'gender': 0, 'occupation': 2, 'work_mode': 1,
            'work_screen_hours': 8.0, 'leisure_screen_hours': 2.0,
        }
    },
    {
        'name': 'Happy student after good results',
        'text': 'I got my results today and I topped the class! Everything feels amazing right now',
        'behavioral': {
            'sleep_hours': 8.0, 'screen_time_hours': 4.0,
            'productivity_0_100': 90, 'exercise_minutes_per_week': 150,
            'social_hours_per_week': 8.0, 'sleep_quality_1_5': 5,
            'age': 21, 'gender': 1, 'occupation': 2, 'work_mode': 1,
            'work_screen_hours': 3.0, 'leisure_screen_hours': 1.0,
        }
    },
    {
        'name': 'Lonely student feeling sad',
        'text': 'I feel so alone here, I have no friends and nothing feels worth doing anymore',
        'behavioral': {
            'sleep_hours': 6.0, 'screen_time_hours': 8.0,
            'productivity_0_100': 20, 'exercise_minutes_per_week': 0,
            'social_hours_per_week': 0.5, 'sleep_quality_1_5': 2,
            'age': 19, 'gender': 0, 'occupation': 2, 'work_mode': 1,
            'work_screen_hours': 4.0, 'leisure_screen_hours': 4.0,
        }
    },
]

print('=' * 55)
print('  SCENARIO TESTING — 3 Student Profiles')
print('=' * 55)

emotion_emoji = {
    'angry': '😠', 'disgust': '🤢', 'fear': '😨',
    'happy': '😊', 'neutral': '😐', 'sad': '😢', 'surprise': '😲'
}

for scenario in scenarios:
    print(f'\n🔍 Scenario: {scenario["name"]}')
    print(f'   Text: "{scenario["text"][:60]}"')

    t_probs = get_text_probs(scenario['text'])
    b_probs = get_behavioral_probs(scenario['behavioral'])

    # Use dummy face and voice for scenarios
    f_probs = align_to_emotions(
        facial_model.predict(np.random.rand(1,96,96,3).astype('float32'), verbose=0)[0],
        facial_labels, EMOTIONS
    )
    v_probs = align_to_emotions(
        voice_model.predict(np.random.rand(1,1024).astype('float32'), verbose=0)[0],
        voice_labels, EMOTIONS
    )

    result = fuse(f_probs, v_probs, t_probs, b_probs)
    e      = result['final_emotion']
    emoj   = emotion_emoji.get(e, '')

    print(f'   {emoj} Detected: {e.upper()} ({result["confidence"]}% confidence)')
    top3 = sorted(result['fused_scores'].items(), key=lambda x: -x[1])[:3]
    print(f'   Top 3: {" | ".join([f"{emotion_emoji.get(em,"")}{em}:{round(sc*100)}%" for em,sc in top3])}')

print('\n✅ All scenarios tested!')

  SCENARIO TESTING — 3 Student Profiles

🔍 Scenario: Stressed student before exams
   Text: "I cannot sleep, my exams are tomorrow and I feel like I will"
   😨 Detected: FEAR (33.28% confidence)
   Top 3: 😨fear:33% | 😠angry:30% | 😢sad:22%

🔍 Scenario: Happy student after good results
   Text: "I got my results today and I topped the class! Everything fe"
   😠 Detected: ANGRY (32.41% confidence)
   Top 3: 😠angry:32% | 😨fear:31% | 😲surprise:18%

🔍 Scenario: Lonely student feeling sad
   Text: "I feel so alone here, I have no friends and nothing feels wo"
   😨 Detected: FEAR (33.4% confidence)
   Top 3: 😨fear:33% | 😠angry:32% | 😢sad:22%

✅ All scenarios tested!


In [22]:
# ============================================================
# CELL 15 — Text-Only Quick Test
# Type any sentence and instantly see emotion prediction
# ============================================================

# ✏️ Change these sentences to test anything you want
test_sentences = [
    "I am so happy today, everything is going great!",
    "I feel really sad and lonely, nothing is working out",
    "I am so angry, this is completely unfair",
    "I am terrified about my presentation tomorrow",
    "Today was just a normal day, nothing much happened",
    "I cannot believe I got selected for the internship!",
    "The way things are done here is absolutely disgusting",
]

emotion_emoji = {
    'angry': '😠', 'disgust': '🤢', 'fear': '😨',
    'happy': '😊', 'neutral': '😐', 'sad': '😢', 'surprise': '😲'
}

print('TEXT-ONLY QUICK TEST (BERT predictions)')
print('=' * 65)
for sentence in test_sentences:
    probs     = predict_bert(sentence)
    predicted = text_labels[np.argmax(probs)]
    conf      = max(probs) * 100
    emoj      = emotion_emoji.get(predicted, '')
    bar       = '█' * int(conf / 5)
    print(f'{emoj} {predicted:<10} {conf:5.1f}%  {bar}')
    print(f'   "{sentence}"')
    print()

TEXT-ONLY QUICK TEST (BERT predictions)
😊 happy       91.4%  ██████████████████
   "I am so happy today, everything is going great!"

😢 sad         91.1%  ██████████████████
   "I feel really sad and lonely, nothing is working out"

😠 angry       90.8%  ██████████████████
   "I am so angry, this is completely unfair"

😨 fear        91.3%  ██████████████████
   "I am terrified about my presentation tomorrow"

😐 neutral     70.2%  ██████████████
   "Today was just a normal day, nothing much happened"

😲 surprise    90.6%  ██████████████████
   "I cannot believe I got selected for the internship!"

🤢 disgust     90.0%  █████████████████
   "The way things are done here is absolutely disgusting"



In [23]:
# ============================================================
# CELL 16 — Save Final Pipeline Summary
# ============================================================
summary = {
    'project':    'MindSight: Generative AI-Powered Student Mental Health Detection and Insights',
    'pipeline_status': 'WORKING END-TO-END ✅',
    'models': {
        'facial': {
            'architecture': 'MobileNetV2 + Dense Head (Transfer Learning)',
            'dataset':      'Balanced FER Dataset (45,304 images, 7 classes)',
            'accuracy':     '84.60%',
            'file':         'facial_model.keras'
        },
        'voice': {
            'architecture': 'YAMNet embeddings + Dense Network',
            'dataset':      'TESS Toronto Emotional Speech Set (2,800 audio files)',
            'accuracy':     '96.96%',
            'file':         'voice_model.keras'
        },
        'text': {
            'architecture': 'BERT-base-uncased fine-tuned',
            'dataset':      'dair-ai/emotion + GoEmotions (filtered)',
            'file':         'bert_text_model_v3/'
        },
        'behavioral': {
            'architecture': 'Random Forest Classifier',
            'dataset':      'ScreenTime vs MentalWellness + Digital Behavior + Mobile Data',
            'accuracy':     '91.25%',
            'file':         'behavioral_model.pkl'
        }
    },
    'fusion': {
        'method':  'Weighted average (MindLift paper weights)',
        'weights': WEIGHTS,
        'output':  '7 emotions: angry, disgust, fear, happy, neutral, sad, surprise'
    },
    'report_generation': 'Groq LLaMA 3.3 70B via Groq API'
}

with open(f'{SAVE_DIR}/final_pipeline_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('✅ Pipeline summary saved to Drive!')
print(json.dumps(summary, indent=2))

✅ Pipeline summary saved to Drive!
{
  "project": "MindSight: Generative AI-Powered Student Mental Health Detection and Insights",
  "pipeline_status": "WORKING END-TO-END \u2705",
  "models": {
    "facial": {
      "architecture": "MobileNetV2 + Dense Head (Transfer Learning)",
      "dataset": "Balanced FER Dataset (45,304 images, 7 classes)",
      "accuracy": "84.60%",
      "file": "facial_model.keras"
    },
    "voice": {
      "architecture": "YAMNet embeddings + Dense Network",
      "dataset": "TESS Toronto Emotional Speech Set (2,800 audio files)",
      "accuracy": "96.96%",
      "file": "voice_model.keras"
    },
    "text": {
      "architecture": "BERT-base-uncased fine-tuned",
      "dataset": "dair-ai/emotion + GoEmotions (filtered)",
      "file": "bert_text_model_v3/"
    },
    "behavioral": {
      "architecture": "Random Forest Classifier",
      "dataset": "ScreenTime vs MentalWellness + Digital Behavior + Mobile Data",
      "accuracy": "91.25%",
      "file":